In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import re
from urllib.parse import urlunsplit, urlencode

In [ ]:
def fetch_holidays(year: int, country_iso: str) -> list[dict]:
    SCHEME = 'https'
    BASE= 'date.nager.at/api'
    PATH = f'/v3/PublicHolidays/{year}/{country_iso}'
    def get_url(path,query_dict=None):
        if query_dict is None:
            query_dict = {}
        query = urlencode(query_dict)
        return urlunsplit((SCHEME,BASE,PATH,query,''))
    response = requests.get(get_url(PATH))
    return response.json()

In [ ]:
fetch_holidays(2024, "ES")

In [ ]:
def transform_holidays(response_json: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(response_json)
    new = []
    for row in df.iterrows():
        new_row = row.copy()
        comunidades = row['counties']
        for comunidad in comunidades:
            new_row['comunidad'] = comunidad
            new.append(new_row)

    return None

In [ ]:
df = pd.DataFrame(fetch_holidays(2024, "ES"))
new = []
df_final = df.copy()
df_final.loc[df_final['counties'].isna(),'counties'] = 'todos'
df2 = df_final.explode('counties').groupby('counties')['localName'].count().reset_index()
df2.loc[df['counties']!='todos','localName'] += 10
df2.drop(17)

In [ ]:
def scrape_comunities(url):
    response = requests.get(url)
    response.content

In [ ]:
import io
url = 'https://es.wikipedia.org/wiki/ISO 3166-2:ES'
response = requests.get(url)
res = BeautifulSoup(response.content)
tabla = res.find('table')
df = pd.read_html(str(tabla))[0]
df = df.rename(columns={'Código':'counties'})

In [ ]:
df_final = df2.merge(df, on = 'counties', how = 'inner')
df_final

In [ ]:
def max_holidays_community(df:pd.DataFrame):
    return df.loc[df['localName']>= df['localName'].max(),'counties']
max_holidays_community(df_final)


In [ ]:
def min_holidays_community(df:pd.DataFrame):
    return df.loc[df['localName']<= df['localName'].min(),'localName']
min_holidays_community(df_final)

PROBLEMA 2

In [ ]:
df_cardio = pd.read_csv('cardiovascular.csv')
df_cardio

In [ ]:
df = df_cardio.copy()
df['Income'].isna().sum()

In [ ]:
df['Income'] = (df_cardio.groupby('Education')['Income'].transform(lambda m: m.fillna(m.median())))
df['Income'].isna().sum()

In [ ]:
df['zscore'] = abs((df['BMI']-df['BMI'].mean())/df['BMI'].std())
df.loc[df['zscore']>3]

In [ ]:
df['BMI'].plot(kind='box')

In [ ]:
patron = re.compile(r'\(?\+?(?:00)?(\d{1,})\)?\s?\-?(\d{9})')
def cambio(celda):
    
    m = patron.search(celda)
    if m is not None:
        return f'{m.group(1)},{m.group(2)}'
df['Phone_Number']=df['Phone_Number'].apply(cambio)
# después de apply(cambio) …
cols = (df["Phone_Number"]
        .str.split(",", expand=True))
df[["Prefix", "Number"]] = cols
df